# Prepare picks for inversion

In [1]:
import numpy as np
import pandas as pd
import pyvista as pv

import glob, os

import pygimli as pg
import pygimli.meshtools as mt
from pygimli.physics import traveltime as tt
from pygimli.viewer.pv import drawSensors,drawSlice
from pygimli.physics import TravelTimeManager

import matplotlib.pylab as plt
from sgt_file_generator import create_sgt_file
from read_picks import load_picks, calculate_dt

%matplotlib inline 
#%matplotlib widget
%load_ext autoreload
%autoreload 2

### Load in all the picks

In [ ]:
path_to_pick_files = glob.glob('picks/picks_2025/HS*.txt')
path_to_pick_files += glob.glob('picks/picks_2024/HS*.txt')
path_to_pick_files.sort()

path_to_strikewise_folder = 'strike_wise'

In [ ]:
all_picks = load_picks(path_to_pick_files, path_to_strikewise_folder)

print("Number of all picks collected:", len(all_picks))
print("Number of unique experiments:", len(all_picks['experiment'].unique()))
print("\tUnique experiments:", all_picks['experiment'].unique())
print("Number of unique stations:", len(all_picks['sta'].unique()))

### Overviewplot of loaded sources and receivers

In [ ]:
plt.figure()

plt.scatter(all_picks['sta_lon'], all_picks['sta_lat'], c='b', marker='v', label='Stations')
for idx, row in all_picks[['sta_lon', 'sta_lat', 'sta']].drop_duplicates().iterrows():
    plt.text(row['sta_lon'], row['sta_lat'], row['sta'], fontsize=8, color='darkblue', alpha=0.7)

plt.scatter(all_picks['ev_lon'], all_picks['ev_lat'], c='r', marker='*', label='Strikes')
for idx, row in all_picks[['ev_lon', 'ev_lat', 'experiment']].drop_duplicates().iterrows():
    plt.text(row['ev_lon'], row['ev_lat'], int(row['experiment']), fontsize=8, color='k', alpha=0.7)

plt.xlabel('Longitude')
plt.ylabel('Latitude')

plt.grid('both') 

plt.legend()
plt.show()

### Calculate dt for the hammer strikes

In [ ]:
df_picks = calculate_dt(all_picks)
df_picks['dt'] = df_picks['dt']

### Sanity plots

In [ ]:
# df_pick_exp1 = calculate_dt(all_picks[all_picks['experiment'] == 1])
# selected_hs = 1

# plt.figure()

# df_hammerstrike = df_pick_exp1[(df_pick_exp1['experiment'] == selected_hs) & (df_pick_exp1['phase'] == 'P')]
# plt.scatter(df_hammerstrike['distance_to_source'], df_hammerstrike['dt'], c='b', marker='o', label=f'Hammer strike {selected_hs}')  

# grouped = df_hammerstrike.groupby('distance_to_source')['dt'].agg(['mean', 'std']).reset_index()
# plt.errorbar(grouped['distance_to_source'], grouped['mean'], yerr=grouped['std'], color='g', fmt='o', linestyle='-', capsize=5, alpha=0.5, label='combined with errorbar')

# print(f"Number of picks for hammer strike {selected_hs}:", len(df_hammerstrike))
# print(f"Number of groups for hammer strike {selected_hs}:", len(df_hammerstrike['group'].unique()))

# # plt.ylim (-0.001, 0.04)
# # plt.xlim (0, 10)
# plt.xlabel('Distance from hammerstrike [m] ')
# plt.ylabel('dt [s]')
# plt.title(f'Hammer strike {selected_hs} | dt vs. dx')
# plt.legend()
# plt.grid(True)
# plt.show()

In [ ]:
plt.figure()

# lon = df_picks['distance_to_source']
lon = df_picks['sta_lon']
dt = df_picks['dt']
col = df_picks['experiment']

plt.scatter(lon, dt, s=50, c=col, alpha=0.3, label='Experiment 1')

plt.axhline(0, color='k', linestyle='--', alpha=0.5)
plt.colorbar(label='Experiment Number')
plt.title('All in one: dt vs. lon')
plt.xlabel('lon [m] ')
plt.ylabel('dt [s]')
plt.show()

In [ ]:
df_picks['sta'].unique()

In [ ]:
selected_sta = '31594'

plt.figure()

df_station = df_picks[(df_picks['sta'] == selected_sta) & (df_picks['phase'] == 'P')]

print(f"Number of picks for station {selected_sta}:", len(df_station))
print(f"Number of groups for station {selected_sta}:", len(df_station['group'].unique()))

plt.scatter(df_station['distance_to_source'], df_station['dt'], c='b', marker='o', label=f'Station {selected_sta}')  

grouped = df_station.groupby('distance_to_source')['dt'].agg(['mean', 'std']).reset_index()
plt.errorbar(grouped['distance_to_source'], grouped['mean'], yerr=grouped['std'], color='g', fmt='o', linestyle='-', capsize=5, alpha=0.5, label='combined with errorbar')


plt.xlabel('Distance from hammerstrike [m] ')
plt.ylabel('dt [s]')
plt.title(f'Station {selected_sta} | dt vs. dx')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
selected_hs = 1
selected_hs2= 21

plt.figure()

df_hammerstrike = df_picks[(df_picks['experiment'] == selected_hs) & (df_picks['phase'] == 'P')]
plt.scatter(df_hammerstrike['distance_to_source'], df_hammerstrike['dt'], c='b', marker='o', label=f'Hammer strike {selected_hs}')  

grouped = df_hammerstrike.groupby('distance_to_source')['dt'].agg(['mean', 'std']).reset_index()
plt.errorbar(grouped['distance_to_source'], grouped['mean'], yerr=grouped['std'], color='g', fmt='o', linestyle='-', capsize=5, alpha=0.5, label='combined with errorbar')

print(f"Number of picks for hammer strike {selected_hs}:", len(df_hammerstrike))
print(f"Number of groups for hammer strike {selected_hs}:", len(df_hammerstrike['group'].unique()))

df_hammerstrike2 = df_picks[(df_picks['experiment'] == selected_hs2) & (df_picks['phase'] == 'S')]
print(f"Number of picks for hammer strike {selected_hs2}:", len(df_hammerstrike2))
print(f"Number of groups for hammer strike {selected_hs2}:", len(df_hammerstrike2['group'].unique()))

plt.scatter(df_hammerstrike2['distance_to_source'], df_hammerstrike2['dt'], c='r', marker='o', label=f'Hammer strike {selected_hs2}')  

grouped2 = df_hammerstrike2.groupby('distance_to_source')['dt'].agg(['mean', 'std']).reset_index()
plt.errorbar(grouped2['distance_to_source'], grouped2['mean'], yerr=grouped2['std'], color='m', fmt='o', linestyle='-', capsize=5, alpha=0.5, label='combined with errorbar')
# 
# plt.ylim (-0.001, 0.04)
# plt.xlim (0, 10)
plt.xlabel('Distance from hammerstrike [m] ')
plt.ylabel('dt [s]')
plt.title(f'Hammer strikes| dt vs. dx')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

fig, axs = plt.subplots(2, 1, figsize=(12, 8))

## Attention I am filtering out negative dt picks here
df_picks_filtered = df_picks[df_picks['dt'] >= 0.0]
df_picks_filtered['dt'] = df_picks_filtered['dt'].replace(0.0, 0.0001)

for experiment in df_picks_filtered['experiment'].unique():

    selected_hs = experiment

    df_hammerstrike = df_picks_filtered[(df_picks_filtered['experiment'] == selected_hs) & (df_picks_filtered['phase'] == 'P')]
    # plt.scatter(df_hammerstrike['sta_lon'], df_hammerstrike['dt'], c='b', marker='o', label=f'{len(df_hammerstrike)} | {len(df_hammerstrike['group'].unique())} ')  

    grouped_lon = df_hammerstrike.groupby('sta_lon')['dt'].agg(['mean', 'std']).reset_index()
    axs[0].errorbar(grouped_lon['sta_lon'], grouped_lon['mean'], yerr=grouped_lon['std'], color='g', fmt='o', linestyle='-', capsize=5, alpha=0.5, label='combined with errorbar')
    
    grouped_dist = df_hammerstrike.groupby('distance_to_source')['dt'].agg(['mean', 'std']).reset_index()
    axs[1].errorbar(grouped_dist['distance_to_source'], grouped_dist['mean'], yerr=grouped_dist['std'], color='g', fmt='o', linestyle='-', capsize=5, alpha=0.5, label='combined with errorbar')


axs[0].axhline(0, color='k', linestyle='--', alpha=0.5)
axs[1].axhline(0, color='k', linestyle='--', alpha=0.5)

axs[0].set_xlabel('Longitude [m]')
axs[1].set_xlabel('Distance from hammerstrike [m] ')

axs[0].set_ylabel('dt [s]')
axs[1].set_ylabel('dt [s]')

plt.suptitle(f'All hammer strikes| dt vs. dx')
plt.grid(True)
plt.show()

In [ ]:
# Static 3D plot with dt-based coloring - FIXED VERSION
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Turn off interactive mode temporarily
plt.ioff()

# Create figure with static backend
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Get all unique experiments
all_experiments = sorted(df_picks['experiment'].unique())

print(f"Creating static 3D plot for {len(all_experiments)} hammer strikes")

# Collect all data first
all_stations_x = []
all_stations_y = []
all_stations_z = []
all_sources_x = []
all_sources_y = []
all_source_colors = []

for i, exp in enumerate(all_experiments):
    try:
        # Filter data for this experiment
        df_exp = df_picks[(df_picks['experiment'] == exp) & (df_picks['phase'] == 'P')]
        
        if len(df_exp) == 0:
            continue
        
        # Group by station location and take mean dt
        grouped_exp = df_exp.groupby(['sta_lon', 'sta_lat'])['dt'].mean().reset_index()
        
        if len(grouped_exp) == 0:
            continue
        
        # Collect station data
        all_stations_x.extend(grouped_exp['sta_lon'].tolist())
        all_stations_y.extend(grouped_exp['sta_lat'].tolist())
        all_stations_z.extend(grouped_exp['dt'].tolist())
        
        # Collect source data
        source_info = df_exp[['ev_lon', 'ev_lat']].iloc[0]
        all_sources_x.append(source_info['ev_lon'])
        all_sources_y.append(source_info['ev_lat'])
        all_source_colors.append(grouped_exp['dt'].mean())  # Mean dt for this experiment
        
        print(f"Processed HS {exp}: {len(grouped_exp)} stations")
        
    except Exception as e:
        print(f"Error processing HS {exp}: {e}")
        continue

# Convert to numpy arrays
all_stations_x = np.array(all_stations_x)
all_stations_y = np.array(all_stations_y)
all_stations_z = np.array(all_stations_z)
all_sources_x = np.array(all_sources_x)
all_sources_y = np.array(all_sources_y)
all_source_colors = np.array(all_source_colors)

# Determine color scale
dt_min = min(all_stations_z.min(), all_source_colors.min())
dt_max = max(all_stations_z.max(), all_source_colors.max())
print(f"Travel time range: {dt_min:.6f} to {dt_max:.6f} seconds")

# Create single scatter plot for all stations
scatter_stations = ax.scatter(all_stations_x, all_stations_y, all_stations_z, 
                             c=all_stations_z, cmap='viridis', s=50, alpha=0.8, 
                             edgecolors='white', linewidth=0.3, 
                             vmin=dt_min, vmax=dt_max)

# Create single scatter plot for all sources
scatter_sources = ax.scatter(all_sources_x, all_sources_y, np.zeros(len(all_sources_x)),
                            c=all_source_colors, cmap='viridis', marker='*', s=150, alpha=1.0,
                            edgecolors='black', linewidth=1,
                            vmin=dt_min, vmax=dt_max)

# Add colorbar using the stations scatter (which should work)
try:
    cbar = plt.colorbar(scatter_stations, ax=ax, shrink=0.6, aspect=20, pad=0.1)
    cbar.set_label('Travel Time (dt) [s]', rotation=270, labelpad=20)
except Exception as e:
    print(f"Colorbar error: {e}")
    # Add a manual colorbar
    sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=dt_min, vmax=dt_max))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.6, aspect=20, pad=0.1)
    cbar.set_label('Travel Time (dt) [s]', rotation=270, labelpad=20)

# Set labels and title
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude') 
ax.set_zlabel('Travel Time (dt) [s]')
ax.set_title(f'3D Travel Times - {len(all_experiments)} Hammer Strikes\n(Colors based on Travel Time)')

# Add text box with statistics
textstr = f'Travel Time Range:\n{dt_min:.4f} - {dt_max:.4f} s\nTotal Measurements: {len(all_stations_z)}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text2D(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10,
          verticalalignment='top', bbox=props)

# Save and show
plt.tight_layout()
plt.savefig('3d_travel_times_colored_fixed.png', dpi=150, bbox_inches='tight')
plt.show()

# Turn interactive mode back on
plt.ion()

print(f"Fixed plot complete! {len(all_experiments)} experiments processed")
print(f"Total stations plotted: {len(all_stations_z)}")
print(f"Total sources plotted: {len(all_sources_x)}")
print(f"Color scale: {dt_min:.6f} to {dt_max:.6f} seconds")

In [ ]:
def fit_plane_to_points(x, y, z):
    """Fit a plane of the form z = ax + by + c to the data points"""
    # Create design matrix
    A = np.column_stack([x, y, np.ones(len(x))])
    
    # Solve least squares: A * [a, b, c] = z
    coeffs, residuals, rank, s = np.linalg.lstsq(A, z, rcond=None)
    
    return coeffs  # [a, b, c] where z = ax + by + c

def plot_all_hammer_strike_planes():
    """Plot fitted planes for all hammer strikes in one plot"""
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    
    all_experiments = sorted(df_picks['experiment'].unique())
    colors = plt.cm.tab20(np.linspace(0, 1, len(all_experiments)))
    
    for i, exp in enumerate(all_experiments):
        df_exp = df_picks[(df_picks['experiment'] == exp) & (df_picks['phase'] == 'P')]
        
        if len(df_exp) < 3:  # Need at least 3 points to fit a plane
            continue
            
        # Group by station location
        grouped_exp = df_exp.groupby(['sta_lon', 'sta_lat'])['dt'].mean().reset_index()
        
        if len(grouped_exp) < 3:
            continue
        
        x = grouped_exp['sta_lon'].values.astype('float')
        y = grouped_exp['sta_lat'].values.astype('float')
        z = grouped_exp['dt'].values.astype('float')
        
        try:
            # Fit plane
            a, b, c = fit_plane_to_points(x, y, z)
            
            # Create smaller grid for this plane
            x_range = np.linspace(x.min(), x.max(), 10)
            y_range = np.linspace(y.min(), y.max(), 10)
            X_grid, Y_grid = np.meshgrid(x_range, y_range)
            Z_grid = a * X_grid + b * Y_grid + c
            
            # Plot plane
            ax.plot_surface(X_grid, Y_grid, Z_grid, 
                           alpha=0.3, color=colors[i], 
                           label=f'HS {exp} plane')
            
            # Plot data points
            # ax.scatter(x, y, z, c=[colors[i]], s=30, alpha=0.8)
            
            # print(f"HS {exp}: z = {a:.6f}*x + {b:.6f}*y + {c:.6f}")
            source_info = df_exp[['ev_lon', 'ev_lat']].iloc[0]
            ax.scatter([source_info['ev_lon']], [source_info['ev_lat']], [0], 
                  c=[colors[i]], marker='*', s=120, alpha=1.0, 
                  edgecolors='black', linewidth=1)

            
        except Exception as e:
            print(f"Could not fit plane for HS {exp}: {e}")
            continue
    
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_zlabel('Travel Time (dt) [s]')
    ax.set_title('Fitted planes for all hammer strikes')
    plt.show()

# Run the multi-plane plot
plot_all_hammer_strike_planes()

## Generating SGT files for pyGIMLi

In [ ]:
# CORRECTED: Create unified coordinate list and proper indexing
import pandas as pd
import numpy as np

# Step 0: get rid of negative dt picks
df_picks = df_picks[df_picks['dt'] >= 0.0].reset_index(drop=True)
df_picks['dt'] = df_picks['dt'].replace(0.0, 0.0001)

# Step 1: Create unified coordinate list (sources first, then receivers)
sources = df_picks[['ev_lon', 'ev_lat']].drop_duplicates().reset_index(drop=True)
sources = sources.rename(columns={'ev_lon': 'x', 'ev_lat': 'y'})
sources['coord_type'] = 'source'

receivers = df_picks[['sta_lon', 'sta_lat']].drop_duplicates().reset_index(drop=True) 
receivers = receivers.rename(columns={'sta_lon': 'x', 'sta_lat': 'y'})
receivers['coord_type'] = 'receiver'

# Combine into single coordinate list (SGT format requirement)
all_coordinates = pd.concat([sources[['x', 'y']], receivers[['x', 'y']]], 
                           ignore_index=True)

print(f"Total coordinates: {len(all_coordinates)}")
print(f"Sources: {len(sources)} (indices 0-{len(sources)-1})")
print(f"Receivers: {len(receivers)} (indices {len(sources)}-{len(all_coordinates)-1})")

# Step 2: Create mapping dictionaries for the unified index space
source_coord_to_idx = {}
for idx, row in sources.iterrows():
    coord_key = (row['x'], row['y'])
    source_coord_to_idx[coord_key] = idx  # Sources get indices 0, 1, 2, ...

receiver_coord_to_idx = {}
for idx, row in receivers.iterrows():
    coord_key = (row['x'], row['y'])
    receiver_coord_to_idx[coord_key] = idx + len(sources)  # Receivers get indices starting after sources

print(f"Source index range: 0 to {len(sources)-1}")
print(f"Receiver index range: {len(sources)} to {len(all_coordinates)-1}")

In [ ]:
# Step 3: Create travel times for ONLY the measured pairs (no duplication)
travel_times_indexed = []

for idx, row in df_picks[['ev_lon', 'ev_lat', 'sta_lon', 'sta_lat', 'dt']].iterrows():
    # Get source coordinates
    source_coord = (row['ev_lon'], row['ev_lat'])
    # Get receiver coordinates  
    receiver_coord = (row['sta_lon'], row['sta_lat'])
    
    # Look up indices in unified coordinate system
    source_idx = source_coord_to_idx.get(source_coord)
    receiver_idx = receiver_coord_to_idx.get(receiver_coord)
    
    # Only add if both indices found
    if source_idx is not None and receiver_idx is not None:
        travel_times_indexed.append([
            source_idx,      # source index (0 to len(sources)-1)
            receiver_idx,    # receiver index (len(sources) to len(all_coords)-1)  
            row['dt'],       # travel time
            1               # valid flag (1 = valid, 0 = invalid)
        ])

# Convert to numpy array and remove duplicates
travel_times_df = pd.DataFrame(travel_times_indexed, 
                              columns=['source_idx', 'receiver_idx', 'dt', 'valid'])
travel_times_unique = travel_times_df.drop_duplicates(subset=['source_idx', 'receiver_idx'])
travel_times_array = travel_times_unique.values

print(f"Original travel time records: {len(df_picks)}")
print(f"After coordinate mapping: {len(travel_times_indexed)}")
print(f"After removing duplicates: {len(travel_times_array)}")
print(f"Unique source-receiver pairs: {len(travel_times_unique)}")

print("\nSample travel time entries:")
print("source_idx, receiver_idx, time, valid_flag")
for i in range(min(5, len(travel_times_array))):
    print(f"{travel_times_array[i][0]:8.0f}, {travel_times_array[i][1]:11.0f}, {travel_times_array[i][2]:8.6f}, {travel_times_array[i][3]:10.0f}")
    

In [ ]:
# Step 4: Verify the unified indexing is correct
def verify_unified_indexing():
    print("Verification of unified coordinate indexing:")
    print("-" * 60)
    
    # Check source indices
    print("Source coordinates (first few):")
    for i in range(min(3, len(sources))):
        coord = (sources.iloc[i]['x'], sources.iloc[i]['y'])
        mapped_idx = source_coord_to_idx[coord]
        print(f"  Index {mapped_idx}: {coord} (should be < {len(sources)})")
    
    print()
    
    # Check receiver indices  
    print("Receiver coordinates (first few):")
    for i in range(min(3, len(receivers))):
        coord = (receivers.iloc[i]['x'], receivers.iloc[i]['y'])
        mapped_idx = receiver_coord_to_idx[coord]
        print(f"  Index {mapped_idx}: {coord} (should be >= {len(sources)})")
    
    print()
    
    # Check travel time pairs
    print("Travel time pair verification (first few):")
    for i in range(min(3, len(travel_times_array))):
        source_idx = int(travel_times_array[i][0])
        receiver_idx = int(travel_times_array[i][1])
        
        # Get coordinates from unified list
        source_coords = (all_coordinates.iloc[source_idx]['x'], 
                        all_coordinates.iloc[source_idx]['y'])
        receiver_coords = (all_coordinates.iloc[receiver_idx]['x'], 
                          all_coordinates.iloc[receiver_idx]['y'])
        
        print(f"  Source {source_idx}: {source_coords}")
        print(f"  Receiver {receiver_idx}: {receiver_coords}")
        print(f"  Travel time: {travel_times_array[i][2]:.6f}")
        print()

verify_unified_indexing()

In [ ]:
# Step 5: Create SGT file with correct unified indexing
create_sgt_file(
    sources=all_coordinates,  # Now contains ALL coordinates (sources + receivers)
    receivers=pd.DataFrame(columns=['x', 'y']),  # Empty - not used when sources contains everything
    travel_times=travel_times_array,  # Only actual measured pairs
    filename='saintsmology_corrected.sgt'
)

print("Corrected SGT file created!")
print(f"Total coordinates in file: {len(all_coordinates)}")
print(f"Travel time measurements: {len(travel_times_array)}")
print(f"Index ranges:")
print(f"  Sources: 1 to {len(sources)} (in SGT 1-based indexing)")
print(f"  Receivers: {len(sources)+1} to {len(all_coordinates)} (in SGT 1-based indexing)")